## Data Ingestion

O Pipilene de ingestão de dados recebe o arquivo complaints.csv.zip obtido desta no site da **CFPB Open Tech** neste [link](https://www.consumerfinance.gov/data-research/consumer-complaints/#get-the-data)
e:
1) Verifica se já existe o arquivo final (idempotência)
2) Extrai o CSV do ZIP
3) Detecta automaticamente o encoding do CSV
4) Usa DuckDB para ler, filtrar e transformar os dados
5) Grava o resultado em Parquet com compressão
6) Valida a quantidade de registros
7) Remove o CSV temporário
8) Fecha a conexão com o banco

A Saída é o arquivo complaints_lean_2025.parquet com os dados filtrados:
- Apenas do ANO 2025
- Apenas linhas que possuem texto no campo **Consumer complaint narrative**
- Nomes das colunas modificados para padrão snake case

[Voltar para notebook principal](./0_notebook_principal.ipynb)

In [1]:
import zipfile
import os
import duckdb
import chardet
import time
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# --- Configurações e Caminhos ---
BASE_DIR = Path.cwd()
ZIP_PATH = BASE_DIR / "data" / "raw" / "complaints.csv.zip"
EXTRACT_DIR = BASE_DIR / "data" / "raw" / "extracted"
PARQUET_DIR = BASE_DIR / "data" / "parquet"
OUTPUT_FILE = PARQUET_DIR / "complaints_lean_2025.parquet"


def ingest_hybrid_lean():
    print("--- Pipeline Híbrido: Extração Segura + Ingestão Lean (2025) ---")

    # 1. Idempotência: Se o Parquet já existe, não faz nada
    if OUTPUT_FILE.exists():
        print(f"[SKIP] Arquivo final já existe: {OUTPUT_FILE}")
        return

    # Garantir diretórios
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    PARQUET_DIR.mkdir(parents=True, exist_ok=True)

    csv_path = None

    try:
        # --- PASSO 1: Extração Física ---
        print("1. Extraindo CSV do ZIP...")
        start_extract = time.time()

        with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
            csv_files = [f for f in zip_ref.namelist() if f.endswith(".csv")]
            if not csv_files:
                raise FileNotFoundError("Nenhum CSV encontrado no ZIP.")

            target_csv = csv_files[0]
            zip_ref.extract(target_csv, EXTRACT_DIR)
            csv_path = EXTRACT_DIR / target_csv

        print(f"   Extraído: {csv_path} ({time.time() - start_extract:.2f}s)")

        # --- PASSO 2: Detecção de Encoding ---
        print("2. Detectando Encoding...")
        with open(csv_path, "rb") as f:
            result = chardet.detect(f.read(100000))
            detected_encoding = result["encoding"]
            print(f"   Encoding detectado: {detected_encoding}")

        # --- PASSO 3: Transformação e Conversão ---
        print("3. Executando DuckDB (Filtro 2025 + Parquet)...")
        start_convert = time.time()

        con = duckdb.connect()

        query = f"""
            COPY (
                SELECT 
                    CAST("Complaint ID" AS VARCHAR) AS complaint_id,
                    TRY_CAST("Date received" AS DATE) AS date_received,
                    TRIM("Product") AS product,
                    TRIM("Sub-product") AS sub_product,
                    TRIM("Issue") AS issue,
                    TRIM("Sub-issue") AS sub_issue,
                    "Consumer complaint narrative" AS consumer_complaint_narrative,
                    TRIM("Company") AS company,
                    TRIM("Company response to consumer") AS company_response_to_consumer,
                    "Consumer disputed?" AS consumer_disputed,
                    
                    True AS has_narrative
                
                FROM read_csv('{csv_path}', 
                              header=True, 
                              auto_detect=True,
                              encoding='{detected_encoding}',
                              ignore_errors=True)
                
                WHERE 
                    -- Filtro 1: Garantir que existe texto
                    "Consumer complaint narrative" IS NOT NULL
                    
                    AND 
                    
                    -- Filtro 2: Apenas ano de 2025
                    -- Nota: Usamos a conversão direta aqui no WHERE para garantir
                    YEAR(TRY_CAST("Date received" AS DATE)) = 2025

            ) TO '{OUTPUT_FILE}' (FORMAT 'PARQUET', CODEC 'SNAPPY');
        """

        con.sql(query)
        print(f"   Sucesso! Arquivo salvo em: {OUTPUT_FILE}")

        # Validação rápida de contagem
        count = con.sql(f"SELECT COUNT(*) FROM '{OUTPUT_FILE}'").fetchone()[0]
        print(f"   Registros processados: {count}")
        print(f"   Tempo de conversão: {time.time() - start_convert:.2f}s")

    except Exception as e:
        print(f"❌ Erro crítico: {e}")
        if OUTPUT_FILE.exists():
            os.remove(OUTPUT_FILE)

    finally:
        # --- PASSO 4: Limpeza ---
        if csv_path and csv_path.exists():
            print("4. Limpeza: Removendo CSV temporário...")
            try:
                os.remove(csv_path)
            except PermissionError:
                print("   Aviso: Não foi possível remover o CSV (arquivo em uso).")

        if "con" in locals():
            con.close()


if __name__ == "__main__":
    ingest_hybrid_lean()


--- Pipeline Híbrido: Extração Segura + Ingestão Lean (2025) ---
1. Extraindo CSV do ZIP...
   Extraído: F:\workspace\postech\datathon_tech_challenge_5\data\raw\extracted\complaints.csv (138.07s)
2. Detectando Encoding...
   Encoding detectado: ascii
3. Executando DuckDB (Filtro 2025 + Parquet)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Sucesso! Arquivo salvo em: F:\workspace\postech\datathon_tech_challenge_5\data\parquet\complaints_lean_2025.parquet
   Registros processados: 1192432
   Tempo de conversão: 527.66s
4. Limpeza: Removendo CSV temporário...


## Verificação dos dados

In [2]:
import polars as pl

# Carregar Parquet em lazy mode
df = pl.scan_parquet("data/parquet/complaints_lean_2025.parquet")


In [3]:
# Validações
print("Total de registros:", df.select(pl.len()).collect().item())
print("Colunas:", df.collect_schema().names())
print("Categorias de produto:", df.select("product").unique().collect())
print("Período:", df.select([
    pl.col("date_received").min().alias("data_min"),
    pl.col("date_received").max().alias("data_max")
]).collect())

Total de registros: 1192432
Colunas: ['complaint_id', 'date_received', 'product', 'sub_product', 'issue', 'sub_issue', 'consumer_complaint_narrative', 'company', 'company_response_to_consumer', 'consumer_disputed', 'has_narrative']
Categorias de produto: shape: (11, 1)
┌─────────────────────────────────┐
│ product                         │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ Mortgage                        │
│ Prepaid card                    │
│ Debt or credit management       │
│ Credit card                     │
│ Payday loan, title loan, perso… │
│ …                               │
│ Checking or savings account     │
│ Student loan                    │
│ Credit reporting or other pers… │
│ Money transfer, virtual curren… │
│ Debt collection                 │
└─────────────────────────────────┘
Período: shape: (1, 2)
┌────────────┬────────────┐
│ data_min   ┆ data_max   │
│ ---        ┆ ---        │
│ date     

In [4]:
# Verificar valores nulos em campos críticos
null_counts = df.select([
    pl.col("consumer_complaint_narrative").is_null().sum().alias("complaint_text_null"),
    pl.col("product").is_null().sum().alias("product_null"),
    pl.col("company_response_to_consumer").is_null().sum().alias("response_null")
]).collect()

print("Valores nulos:", null_counts)

Valores nulos: shape: (1, 3)
┌─────────────────────┬──────────────┬───────────────┐
│ complaint_text_null ┆ product_null ┆ response_null │
│ ---                 ┆ ---          ┆ ---           │
│ u32                 ┆ u32          ┆ u32           │
╞═════════════════════╪══════════════╪═══════════════╡
│ 0                   ┆ 0            ┆ 0             │
└─────────────────────┴──────────────┴───────────────┘


## [Voltar para notebook principal](./0_notebook_principal.ipynb)